In [ ]:


import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import time
import dask.delayed
from dask.distributed import Client
import dask 
import duckdb
import os

from minio import Minio
from minio.error import S3Error


from importlib import reload

import urllib3
# from keras.preprocessing.sequence import pad_sequences
# from keras.models import Sequential
# from keras.layers import LSTM, Dense, Input
# from keras.utils import to_categorical

# from tsfresh import extract_features, select_features
# from tsfresh.feature_extraction.settings import MinimalFCParameters, EfficientFCParameters

pd.options.mode.chained_assignment = None  # default='warn'

project_name_ahjo = "J7045_Futavis"
pd.set_option('display.max_rows', False)
pd.set_option('display.max_columns', None)


%matplotlib widget 

import matplotlib as mpl
import rwth_colors
from cycler import cycler
cm = 1/2.54  # centimeters in inches



# column-name -> metric unit
metric_dic = {"AH_throughput": "Ah", "Temperature": "$^\circ$C", "Voltage": "V",
              "Capacity": "Ah", "Duration": "days", "SOC": "$\%$", "SOH": "$\%$",
              "Current": "A", "Wh_throughput": "Wh", "Capacity_current": "A"}

# if no color is selected, use the default colors of the cycler, which are:
rwth_colors_cycler_color = [
    rwth_colors.colors[('blue', 100)],
    rwth_colors.colors[('black', 100)],
    rwth_colors.colors[('magenta', 100)],
    rwth_colors.colors[('yellow', 100)],
    rwth_colors.colors[('green', 100)],
    rwth_colors.colors[('bordeaux', 100)],
    rwth_colors.colors[('orange', 100)],
    rwth_colors.colors[('turqoise', 100)],
    rwth_colors.colors[('darkred', 100)],
    rwth_colors.colors[('lime', 100)],
    rwth_colors.colors[('petrol', 100)],
    rwth_colors.colors[('lavender', 100)],
    rwth_colors.colors[('red', 100)],

    rwth_colors.colors[('blue', 50)],
    rwth_colors.colors[('black', 50)],
    rwth_colors.colors[('magenta', 50)],
    rwth_colors.colors[('yellow', 50)],
    rwth_colors.colors[('green', 50)],
    rwth_colors.colors[('bordeaux', 50)],
    rwth_colors.colors[('orange', 50)],
    rwth_colors.colors[('turqoise', 50)],
    rwth_colors.colors[('darkred', 50)],
    rwth_colors.colors[('lime', 50)],
    rwth_colors.colors[('petrol', 50)],
    rwth_colors.colors[('lavender', 50)],
    rwth_colors.colors[('red', 50)]
]

rwth_colors_cycler_linestyle = [
    '-',
    '--',
    '-.'
]


cc = (cycler(linestyle=rwth_colors_cycler_linestyle)
      * cycler(color=rwth_colors_cycler_color))

mpl.rcParams['axes.prop_cycle'] = cc
mpl.rcParams['axes.facecolor'] = 'none'
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['axes.titlesize'] = 11
mpl.rcParams['figure.facecolor'] = 'none'
mpl.rcParams['font.size'] = 11
mpl.rcParams['image.cmap'] = 'turbo'
mpl.rcParams['xtick.labelsize'] = 11
mpl.rcParams['ytick.labelsize'] = 11
mpl.rcParams['legend.fontsize'] = 11


def get_unique_by_column(df, filter_column, filter_value, groupby_column,target_column):
    combined_targets =df[df[filter_column]==filter_value].groupby(groupby_column)[target_column].unique().tolist()
    flat_targets = [item for sublist in combined_targets for item in sublist]
    return np.unique(flat_targets)




    
def check_exception (exception_dict):
    second_try = []

    for key, procedures in exception_dict.items():
        #We found some cells that failed the cluster process. Consider to rerun.
        if any("DOD" in procedure for procedure in procedures):
            second_try.append(key)

        else:
            print(procedures)
    return second_try


minio_endpoint = "optimusprime.isea.rwth-aachen.de:9000"
access_key= "8ms0O8n4gwMia5BpDrYq"
secret_key= "WxDnJMQmUrW8RScViRf0CCTBDDIlZaoIANQgTFWl"
bucket_name= "zho"
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



if access_key and secret_key and minio_endpoint:
    minio_client = Minio(
        minio_endpoint,
        access_key=access_key,
        secret_key=secret_key,
        secure=True,
        cert_check=False,
    )

def export_to_server(df, object_name):

    parquet_buffer = io.BytesIO()
    df.to_parquet(parquet_buffer, index=False)
    parquet_buffer.seek(0)

    try:
        minio_client.put_object(
            bucket_name=bucket_name,
            object_name=object_name,
            data=parquet_buffer,
            length=len(parquet_buffer.getvalue()),
        )
        print("File uploaded successfully.")

    except S3Error as err:
        print("Upload error:", err)

In [ ]:
from util.load_battery_config import BatteryConfigJupyter


# Usage example
if __name__ == "__main__":
    # Create and display the configuration UI
    config_ui = BatteryConfigJupyter()
    config_ui.display()

In [ ]:
target_specimen = None
#target_specimen = ["METABatt_A123_APR18650M1B_213"]


In [ ]:
import dismember.dismember_raw_cell
reload(dismember.dismember_raw_cell)
from dismember.dismember_raw_cell import dismember_raw_cell

import feature_extraction.create_features
reload(feature_extraction.create_features)

from feature_extraction.create_features import create_features
import cluster.model_and_supervise
reload(cluster.model_and_supervise)
from cluster.model_and_supervise import first_layer_HDBSCANModel
from cluster.model_and_supervise import second_layer_HDBSCANModel
from cluster.model_and_supervise import supervised_capacity_filter

from cluster import post_cluster_filter
reload(post_cluster_filter)
from calculate import results_fetching
reload(results_fetching)

from util import bronze_column_filter

List_Cell = glob.glob1(working_path+'\BRONZE_CU', '*.parquet')

Cell_subset = [
    items
    for items in List_Cell
    if not target_specimen or any(target in items for target in target_specimen)
]

procedure_filter = "jri_CU"


Project_Schedule = pd.DataFrame()
processed_count = 0

exception_dict = {}

overwrite_dismember = 1
overwrite_preSILVER = 1
overwrite_SILVER = 1

Work_on_preSILVER = 1
Work_on_Silver = 1
Work_on_Gold = 1

for cell in Cell_subset:

    savepath_df_BRONZE = os.path.join(
        working_path, "BRONZE_CU", cell
    )
    savepath_df_preSILVER = os.path.join(
        working_path, "preSILVER", cell
    )
    savepath_X_SILVER = os.path.join(
        working_path, "with_features_post_labeled", cell.split(".")[0] + ".csv"
    )
    savepath_df_SILVER = os.path.join(
        working_path, "SILVER", cell
    )    

    savepath_df_GOLD = os.path.join(
    r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Checkup-Parquet", type_cell, cell
    )
    
    if (type_cell in cell and "eis" not in cell and not os.path.exists(savepath_df_GOLD)):
        try:
            if Work_on_preSILVER:
                #######################
                # Dismembering and Feature Creation 
                # -> creating preSILVER and feature_pre_labeled
                #######################
                print(f"Working on cell {cell}...")
                print("====== Starting dismembering process ======")

                if overwrite_dismember == 0 and os.path.exists(savepath_df_preSILVER):
                    print(f"Skipping {cell} - dismember file already processed")
                    dismembered_df = bronze_column_filter.bronze_column_filter(savepath_df_preSILVER, cell)

                else:
                    dismembered_df = dismember_raw_cell(cell, savepath_df_BRONZE,savepath_df_preSILVER, MIN_ROWS, PAU_DURATION,V_max,procedure_filter)

                number_programms = dismembered_df.groupby("BM_Programm")["Prozedur"].apply(
                lambda x: x.str.contains("jri_CU", na=False).any()).sum()
                print(f"Number of programs found: {number_programms}")

                # Update HDBSCAN parameters for the 1st layer
                # Set min_cluster_size and min_samples to the number of programs found
                hdbscan_para_layer_1["min_cluster_size"] = int(number_programms-1)
                # The larger the value of min_samples you provide, the more conservative the clustering – more points will be declared as noise
                hdbscan_para_layer_1["min_samples"] = int(1)
                # cluster_selection_epsilon ensures that clusters below the given threshold are not split up any further
                hdbscan_para_layer_1["cluster_selection_epsilon"] = 0.5 #0.6 for whole dataset with aging data

                print("====== Starting feature creation process ======")

                X_unlabeled_features_all, new_count = create_features(
                    dismembered_df, cell, working_path, exception_dict, 
                    V_max, V_min, V_nom, 
                    Nom_Capacity, feature_columns, overwrite_preSILVER
                )

            if Work_on_Silver:
                #########################
                # Start Cluster Analysis
                #########################


                if overwrite_SILVER == 0 and os.path.exists(savepath_X_SILVER) and os.path.exists(savepath_df_SILVER):
                    print(f"Skipping {cell} - already processed")
                    X_clustered_filtered = pd.read_csv(savepath_X_SILVER)
                    df_clustered_filtered = pd.read_parquet(savepath_df_SILVER)



                else:
                    print(f"Proceed with SILVER_Programm {cell}")

                    print("====== Starting first layer clustering ======")
                    first_layer_feature_columns = ["Duration_quartile", "ID"]  # Adjust as needed

                    hdbscan_para = hdbscan_para_layer_1

                    df_clustered_layer_1, X_clustered_layer_1, cluster_means_layer_1,cluster_size_layer_1, exception_dict, count = first_layer_HDBSCANModel(
                        X_unlabeled_features_all, dismembered_df, cell, exception_dict,new_count,first_layer_feature_columns,hdbscan_para)

                    post_filter = post_cluster_filter.cluster_filter(
                        number_programms,
                        qOCV_CRate,
                        Nom_Capacity,
                        V_nom,
                        CAP_Rate,
                        CAP_Type,
                        CAP_Temp,
                        target_pulse_duration,
                        pulse_type,
                        pulse_target_unit,
                    )

                    #########################
                    # Check if the capacity cluster is found
                    #########################

                    capacity_status, df_clustered_filtered, counter = supervised_capacity_filter(X_clustered_layer_1, post_filter, df_clustered_layer_1, cluster_means_layer_1, cluster_size_layer_1, 1)

                    #########################
                    # If not, start second layer clustering searching for capacity cluster
                    #########################
                    print("====== Starting second layer clustering ======")

                    second_layer_feature_columns = ["Current_mean","ID"]
                    if capacity_status:
                        capacity_cluster_layer_1 = df_clustered_filtered
                        df_potential_cap = df_clustered_layer_1[
                            (df_clustered_layer_1["target"].isin(capacity_cluster_layer_1))
                        ]
                        X_potential_cap = (X_clustered_layer_1[
                            X_clustered_layer_1["target"].isin(capacity_cluster_layer_1)]
                        )
                        hdbscan_para = hdbscan_para_layer_2

                        df_clustered_layer_2, X_clustered_layer_2, capacity_cluster, exception_dict, counter, count = second_layer_HDBSCANModel(
                        X_potential_cap, df_potential_cap, cell, exception_dict,new_count,second_layer_feature_columns,hdbscan_para,post_filter,df_clustered_layer_1)

                        X_clustered = cluster.model_and_supervise.merge_target(X_clustered_layer_1, X_clustered_layer_2)
                        df_clustered = df_clustered_layer_2.drop(columns=['target']).merge(X_clustered[["ID", "target"]], on='ID', how='left')

                    else:
                        print(f"Capacity cluster found for {cell}, skipping second layer clustering.")
                        X_clustered = X_clustered_layer_1
                        capacity_cluster = df_clustered_filtered
                        df_clustered = df_clustered_layer_1.drop(columns=['target']).merge(X_clustered[["ID", "target"]], on='ID', how='left')

                    #########################
                    # Export SILVER and feature_post_labeled
                    
                    #########################

                    print("====== Finishing with concating all known clusters ======")
                    
                    X_clustered_filtered = cluster.model_and_supervise.add_pulse_qocv_and_concat(
                        post_filter, cluster_means_layer_1, capacity_cluster, counter, X_clustered
                    )

                    df_clustered_filtered = cluster.model_and_supervise.merge_target(df_clustered, X_clustered_filtered)

                    def convert_to_string(df):
                        for col in df.columns:
                            if df[col].dtype == "object":
                                df[col] = df[col].astype(str)
                        return df
                    df_clustered_filtered = convert_to_string(df_clustered_filtered)
                    df_clustered = convert_to_string(df_clustered)

                    X_clustered_filtered.to_csv(savepath_X_SILVER, index=False)
                    #df_clustered_filtered.to_parquet(savepath_df_SILVER, index=False)

            if Work_on_Gold:
                #########################
                # Calculate Capacity and Pulse 
                #########################

                df_final_1 = results_fetching.calculation(
                    qOCV_CRate,
                    Nom_Capacity,
                    target_pulse_duration,
                    pulse_type,
                    pulse_target_unit,
                    df_clustered_filtered,
                )
                df_result = df_final_1.update_pulse()
                df_result = df_final_1.update_capacity()
                df_result = df_final_1.update_qOCV()

                df_after_filter = df_clustered_filtered.copy()
                df_after_filter.update(df_result)

                #########################
                # Add Label_Procedure 
                #########################

                from visualize import add_test_schedule
                add_test_schedule.add_aging_labels(df_after_filter)

                #########################
                # Export GOLD 
                #########################

                savepath_df_GOLD = os.path.join(
                r"Z:\Forschung\ogP\J8005_BMWK_METABatt\Daten\Checkup-Parquet", type_cell, cell
                )
                print(f"Exporting GOLD file for {savepath_df_GOLD}...")    
                df_after_filter.to_parquet(savepath_df_GOLD, index=False)
                #object_name = f"Metabatt/GOLD/{type_cell}/{cell}"
                #export_to_server(df_after_filter, object_name)

                processed_count += new_count


            # Report exceptions
            print(f"Found clusters for {processed_count} cells successfully")


        except Exception as e:
            print(f"Warning: there was an error: {cell}: {type(e).__name__}: {e}")
            exception_dict[cell] = str(e)
            print(f"Could not find capacity cluster for {len(exception_dict)} cells:")

            continue

In [ ]:
dismembered_df.groupby("ID").head(1)

In [ ]:
exception_dict

In [ ]:
dismembered_df.groupby("ID").head(1)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Original code for Name_prefix
df_results_cap['Name_prefix'] = df_results_cap['Name'].apply(lambda x: x.split("-")[0])

# Create a 3x2 subplot layout
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=["DOD = 20%", "DOD = 40%", "DOD = 60%", "DOD = 80%", "DOD = 100%", ""],
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.13,
    horizontal_spacing=0.08,
    specs=[[{}, {}], [{}, {}], [{}, None]]  # Last row has only one subplot
)

# Define subplot positions for each DOD
subplot_positions = {
    20: (1, 1),
    40: (1, 2),
    60: (2, 1),
    80: (2, 2),
    100: (3, 1)
}

# Function to process data for a specific temperature with a specific color
def add_temperature_data(fig, df_temp, color, temp, row, col):
    for soc in df_temp['SOC'].unique():
        df_soc = df_temp[df_temp['SOC'] == soc]
        
        for name_prefix in df_soc['Name_prefix'].unique():
            df_name = df_soc[df_soc['Name_prefix'] == name_prefix]
            
            for c_rate in df_name['C_Rate'].unique():
                df_subset = df_name[df_name['C_Rate'] == c_rate]
                df_subset = df_subset.sort_values('Ah_throughput')
                
                # Determine marker symbol based on C_Rate
                if c_rate == 0.5:
                    marker_symbol = 'circle'
                elif c_rate == 1.0:
                    marker_symbol = 'square'
                else:
                    marker_symbol = 'diamond'
                
                fig.add_trace(
                    go.Scatter(
                        x=df_subset['Ah_throughput'],
                        y=df_subset['Capacity_py'],
                        mode='lines+markers',
                        name=f"T={temp}°C, SOC={soc}, C={c_rate}",
                        line=dict(color=color),
                        marker=dict(
                            color=color, 
                            symbol=marker_symbol, 
                            size=8
                        ),
                        hovertemplate="<b>Name:</b> %{text}<br>" +
                                     "<b>Ah_throughput:</b> %{x:.2f}<br>" +
                                     "<b>Capacity_py:</b> %{y:.2f}<br>" +
                                     "<b>Temperature:</b> " + str(temp) + "°C<br>" +
                                     "<b>SOC:</b> " + str(soc) + "<br>" +
                                     "<b>C_Rate:</b> " + str(c_rate) + "<br>" +
                                     "<b>Time:</b> %{customdata}<extra></extra>",
                        text=df_subset["Name_prefix"],
                        customdata=df_subset["Time"],
                        showlegend=False
                    ),
                    row=row, col=col
                )

# Process each DOD value
for dod in [20, 40, 60, 80, 100]:
    row, col = subplot_positions[dod]
    
    # Filter data for all three temperatures and this DOD
    df_filtered = df_results_cap[
        (df_results_cap["DOD"] == dod) &
        (df_results_cap["Temperature"].isin([15, 25, 35]))  
    ]
    
    # Process T=15 data (blue)
    df_15 = df_filtered[df_filtered["Temperature"] == 15].sort_values(by=["SOC", "Ah_throughput"])
    add_temperature_data(fig, df_15, "lavender", 15, row, col)
    
    # Process T=35 data (red)
    df_35 = df_filtered[df_filtered["Temperature"] == 35].sort_values(by=["SOC", "Ah_throughput"])
    add_temperature_data(fig, df_35, "mistyrose", 35, row, col)
    

    # Process T=25 data with original Plotly Express approach, but with fixed colors for SOC
    df_25 = df_filtered[df_filtered["Temperature"] == 25].sort_values(by=["SOC", "Ah_throughput"])
    
    # Define a colormap for SOC values
    soc_colors = {
        10: rwth_colors.colors["blue"],
        30: rwth_colors.colors["turqoise"],
        50: rwth_colors.colors["orange"],
        70: rwth_colors.colors["red"],
        90: rwth_colors.colors["darkred"]
    }
    
    # Process each SOC value separately with its fixed color
    for soc in df_25['SOC'].unique():
        df_soc = df_25[df_25['SOC'] == soc]
        fig_25 = px.line(df_soc, x="Ah_throughput", y="Capacity_py",
                        color_discrete_map={str(soc): soc_colors.get(soc, "#000000")},
                        markers=True, line_group="Name_prefix", symbol="C_Rate",
                        hover_data=["SOC", "DOD", "C_Rate", "Time", "Temperature"])
        
        # Add traces to main figure
        for trace in fig_25.data:
            trace.name = f"T=25°C, SOC={soc}"
            trace.line.color = soc_colors.get(soc, "#000000")
            fig.add_trace(trace, row=row, col=col)

# Update subplot axes labels
for dod in [20, 40, 60, 80, 100]:
    row, col = subplot_positions[dod]
    fig.update_xaxes(title_text="Ah Throughput", row=row, col=col)
    fig.update_yaxes(title_text="Capacity", row=row, col=col)

# Update overall layout
fig.update_layout(
    title="Battery Capacity vs Ah Throughput at Different DOD Values",
    legend_title="SOC/C_Rate (Temperature Color Coding)",
    width=1500,
    height=1050,  # Slightly increased height to accommodate the new subplot
    plot_bgcolor="white",
    paper_bgcolor="white"
)

# Find the global min and max for both axes
x_min = float('inf')
x_max = float('-inf')
y_min = float('inf')
y_max = float('-inf')

for dod in [20, 40, 60, 80, 100]:
    df_dod = df_results_cap[df_results_cap["DOD"] == dod]
    
    if not df_dod.empty:
        x_min = min(x_min, df_dod['Ah_throughput'].min())
        x_max = max(x_max, df_dod['Ah_throughput'].max())
        y_min = min(y_min, df_dod['Capacity_py'].min())
        y_max = max(y_max, df_dod['Capacity_py'].max())

# Add some padding to the ranges
x_range = [0, x_max * 1.01]
y_range = [y_min * 0.99, y_max*1.01]
# Update grid lines for all subplots
for dod in [20, 40, 60, 80, 100]:
    row, col = subplot_positions[dod]
    fig.update_xaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor="lightgrey",
        row=row,
        col=col,
        range=x_range
    )
    fig.update_yaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor="lightgrey",
        row=row,
        col=col,
        range=y_range
    )

# Show plot
fig.show()

In [ ]:

# Create figure and axis
fig, ax = plt.subplots(figsize=(10,6))

# Plot each group with a different color/marker
for name, group in df_results_cap.groupby("Name"):
    if (group["Temperature"].iloc[0] == "35") and (group["DOD"].iloc[0] != "100"):
        ax.plot(group["Time"], group["Capacity_py"], 
                marker='o', linestyle='-', label=group["SOC"].iloc[0])

# Add labels and legend
plt.xlabel('Ah_throughput')
plt.ylabel('Capacity_py')
plt.title('Capacity_py vs Ah_throughput by Name')
plt.legend()
plt.grid(True)

# Show plot
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px

# Create a prefix column for grouping
df_results_cap['Name_prefix'] = df_results_cap['Name'].apply(lambda x: x.split("-")[0].split("-")[0])

# Filter the dataframe first to get all rows that match the criteria
filtered_df = df_results_cap[(df_results_cap["Temperature"] == "35") & (df_results_cap["DOD"] == "20")]

# Create the figure with the filtered dataframe
fig = px.line(filtered_df, x="Ah_throughput", y="Capacity_py",
              color="Name_prefix", markers=True,
              title="Capacity vs Time for Batteries (Temperature=35, DOD≠100)",
              hover_data=["SOC","DOD","C_Rate","Time"])

# Update layout
fig.update_layout(
    xaxis_title="Ah_throughput",  # Changed to Ah_throughput as in your original code
    yaxis_title="Capacity_py",
    legend_title="Battery Name",
    # Improve legend placement and size
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.05
    ),    # Make the plot area larger
    height=600,
    width=900,
        margin=dict(r=150)

)

# Show plot
fig.show()

In [ ]:

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 6))

# Get unique name prefixes for color mapping
name_prefixes = df_results_pulse['Name'].apply(lambda x: x.split("-")[0]).unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(name_prefixes)))
prefix_to_color = dict(zip(name_prefixes, colors))

# Plot each group with a different color
for name, group in df_results_pulse.groupby("Name"):
    prefix = name.split("-")[0]
    ax.scatter(group["Ah_throughput"], group["Pulse_py"], 
              color=prefix_to_color[prefix], label=prefix, alpha=0.7)

# Remove duplicate labels
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())

# Add labels and title
plt.xlabel('Ah_throughput')
plt.ylabel('Pulse_py')
plt.title('Pulse_py vs Ah_throughput by Name Prefix')
plt.grid(True, linestyle='--', alpha=0.7)

# Show plot
plt.tight_layout()
plt.show()